# 飞书知识库 (Wiki) 完整工作流验证

> **目标**：验证应用 Token vs User Token 的权限边界,以及知识库文档迁入、层级管理、内容追加的全流程. 
> **场景**：
> 1. 应用 Token 创建知识库(预期失败)→ 权限分享
> 2. User Token 创建知识库 → 外部文档迁入 → 内部创建 → 内容追加 → 层级移动

In [5]:
import os
import json
import time
from datetime import datetime
from pathlib import Path
from feishu_client import FeishuClient, md_to_blocks
# ========== 从项目统一缓存读取 token ==========
OAUTH_CACHE = Path("D:/ALL IN AI/MetaBlog/.data/config/feishu_oauth.json")
if not OAUTH_CACHE.exists():
    raise FileNotFoundError(f"Token 缓存不存在: {OAUTH_CACHE}. 请先运行 manual_get_token.ipynb 获取 token. ")
with open(OAUTH_CACHE, "r", encoding="utf-8") as f:
    oauth_data = json.load(f)
USER_ACCESS_TOKEN = oauth_data.get("access_token")
REFRESH_TOKEN = oauth_data.get("refresh_token")
EXPIRE_AT = oauth_data.get("expire_at", 0)
if not USER_ACCESS_TOKEN:
    raise ValueError("缓存中没有 access_token,请重新获取. ")
remaining = EXPIRE_AT - time.time()
print(f"[Token] 已从 {OAUTH_CACHE} 加载")
print(f"[Token] access_token 剩余有效期: {remaining/60:.0f} 分钟")
# 初始化 FeishuClient,把 user_access_token 传进去
client = FeishuClient(user_access_token=USER_ACCESS_TOKEN)
print("[INIT] FeishuClient ready")
print("       tenant token: " + ("已配置" if client.get_tenant_access_token() else "未配置"))
print("       user token: " + ("已配置" if client.user_access_token else "未配置"))


SyntaxError: f-string: expecting '}' (278488582.py, line 31)

In [2]:
# 常量定义
TEST_MOBILE = "13586820267"
FEISHU_DOMAIN = "https://feishu.cn"
TIMESTAMP = datetime.now().strftime('%m%d_%H%M')

# 全局变量(后续 cell 填充)
APP_WIKI_SPACE_ID = None      # 应用 token 创建的知识库(如果成功)
USER_WIKI_SPACE_ID = None     # user token 创建的知识库
TEST_USER_OPEN_ID = None      # 目标用户 open_id
EXTERNAL_DOC_TOKEN = None     # 外部创建的文档 obj_token
INTERNAL_NODE_A = None        # 内部创建的节点A
INTERNAL_NODE_B = None        # 内部创建的节点B
INTERNAL_NODE_C = None        # 内部创建的节点C

## 第一部分：应用 Token 的权限边界

### 1.1 尝试用应用 Token 创建知识库(预期失败)

In [3]:
print(f"\n[TEST] 应用 Token 创建知识库")
try:
    result = client.create_wiki_space(
        name=f"应用Token测试-{TIMESTAMP}",
        description="测试应用级 Token 是否有权限创建知识库",
        use_user_token=False  # 强制使用 tenant token
    )
    APP_WIKI_SPACE_ID = result["space"]["space_id"]
    print(f"[OK] 意外成功！space_id: {APP_WIKI_SPACE_ID}")
    print(f"     链接: {FEISHU_DOMAIN}/wiki/space/{APP_WIKI_SPACE_ID}")
except Exception as e:
    print(f"[FAIL] 应用 Token 创建知识库失败: {e}")
    print(f"     ✅ 验证了：创建 Wiki 知识库必须使用 user_access_token")


[TEST] 应用 Token 创建知识库
[FAIL] 应用 Token 创建知识库失败: API 错误 99991677: Authentication token expired. Please request a new one. | path=/wiki/v2/spaces
     ✅ 验证了：创建 Wiki 知识库必须使用 user_access_token


### 1.2 用 User Token 创建知识库(正确方式)

In [4]:
print(f"\n[CREATE] User Token 创建知识库")
try:
    result = client.create_wiki_space(
        name=f"用户Token知识库-{TIMESTAMP}",
        description="用户级 Token 创建的完整测试知识库",
        use_user_token=True
    )
    USER_WIKI_SPACE_ID = result["space"]["space_id"]
    print(f"[OK] 知识库创建成功")
    print(f"     space_id: {USER_WIKI_SPACE_ID}")
    print(f"     链接: {FEISHU_DOMAIN}/wiki/space/{USER_WIKI_SPACE_ID}")
except Exception as e:
    print(f"[FAIL] {e}")
    raise


[CREATE] User Token 创建知识库
[FAIL] API 错误 99991677: Authentication token expired. Please request a new one. | path=/wiki/v2/spaces


RuntimeError: API 错误 99991677: Authentication token expired. Please request a new one. | path=/wiki/v2/spaces

### 1.3 更新知识库信息

In [ ]:
print(f"\n[UPDATE] 更新知识库信息")
try:
    client.update_wiki_space(
        space_id=USER_WIKI_SPACE_ID,
        name=f"用户Token知识库-{TIMESTAMP}[已更新]",
        description="已更新描述：测试知识库完整工作流"
    )
    print(f"[OK] 知识库信息已更新")
except Exception as e:
    print(f"[FAIL] {e}")

### 1.4 搜索用户(13586820267)并获取 open_id

In [ ]:
print(f"\n[SEARCH USER] 手机号: {TEST_MOBILE}")
try:
    result = client.api("POST", "/contact/v3/users/batch_get_id",
                        json_data={"mobiles": [TEST_MOBILE], "include_resigned": False})
    users = result.get("data", {}).get("user_list", [])
    if users:
        TEST_USER_OPEN_ID = users[0].get("user_id")
        print(f"[OK] 找到用户")
        print(f"     open_id: {TEST_USER_OPEN_ID}")
    else:
        print(f"[WARN] 未找到用户,尝试 keyword 搜索...")
        # fallback
        result2 = client.api("POST", "/contact/v3/users/find_by_department",
                             json_data={"query": TEST_MOBILE})
        users2 = result2.get("data", {}).get("user_list", [])
        if users2:
            TEST_USER_OPEN_ID = users2[0].get("user_id")
            print(f"[OK] 找到用户: {TEST_USER_OPEN_ID}")
except Exception as e:
    print(f"[FAIL] {e}")

### 1.5 分享知识库权限给用户

In [ ]:
print(f"\n[SHARE] 分享知识库权限")
if not TEST_USER_OPEN_ID:
    print(f"[SKIP] 未找到用户 open_id")
else:
    try:
        # Wiki 知识库添加成员
        payload = {
            "member_type": "user",
            "member_id": TEST_USER_OPEN_ID,
            "perm": "view"
        }
        result = client.api("POST", f"/wiki/v2/spaces/{USER_WIKI_SPACE_ID}/members",
                            json_data=payload, use_user_token=True)
        print(f"[OK] 已分享给用户 {TEST_MOBILE}")
        print(f"     权限: view")
    except Exception as e:
        print(f"[FAIL] {e}")

### 1.6 验证权限列表

In [ ]:
print(f"\n[VERIFY] 知识库成员列表")
try:
    result = client.api("GET", f"/wiki/v2/spaces/{USER_WIKI_SPACE_ID}/members",
                        params={"page_size": "50"}, use_user_token=True)
    members = result.get("data", {}).get("items", [])
    print(f"[OK] 共 {len(members)} 位成员")
    for m in members[:5]:
        print(f"     - {m.get('member_type', '?')} : {m.get('member_id', '?')[:20]}... (perm={m.get('perm', '?')})")
except Exception as e:
    print(f"[FAIL] {e}")

## 第二部分：文档迁入与层级管理

### 2.1 外部创建文档(user token)

In [ ]:
print(f"\n[CREATE EXTERNAL DOC] 外部创建文档(user token)")
try:
    result = client.api("POST", "/docx/v1/documents",
                        json_data={"title": f"外部文档-{TIMESTAMP}"}, use_user_token=True)
    doc = result.get("document", {})
    EXTERNAL_DOC_TOKEN = doc.get("document_id")
    print(f"[OK] 外部文档创建成功")
    print(f"     obj_token: {EXTERNAL_DOC_TOKEN}")
    print(f"     链接: {FEISHU_DOMAIN}/docx/{EXTERNAL_DOC_TOKEN}")
except Exception as e:
    print(f"[FAIL] {e}")

### 2.2 将外部文档迁入知识库

In [ ]:
print(f"\n[MOVE TO WIKI] 迁入外部文档")
try:
    result = client.move_doc_to_wiki(
        space_id=USER_WIKI_SPACE_ID,
        obj_token=EXTERNAL_DOC_TOKEN
    )
    task_id = result.get("task_id")
    if not task_id:
        print(f"[FAIL] 未获取到 task_id: {result}")
    else:
        print(f"[OK] 迁入任务已提交,task_id: {task_id}")
        print(f"     正在轮询任务状态...")
        move_result = client.get_wiki_task(task_id)
        node = move_result.get("node", {})
        EXTERNAL_NODE_TOKEN = node.get("node_token")
        print(f"[OK] 迁入成功")
        print(f"     node_token: {EXTERNAL_NODE_TOKEN}")
        print(f"     obj_token: {node.get('obj_token')}")
        print(f"     链接: {FEISHU_DOMAIN}/wiki/{EXTERNAL_NODE_TOKEN}")
except Exception as e:
    print(f"[FAIL] {e}")
    print(f"     提示：只有 user_access_token 创建的文档才能迁入")

### 2.3 在知识库内部创建文档节点

In [ ]:
print(f"\n[CREATE INTERNAL] 知识库内创建节点")
try:
    # 节点A
    result_a = client.create_wiki_node(
        space_id=USER_WIKI_SPACE_ID,
        title=f"内部文档A-{TIMESTAMP}",
        obj_type="docx"
    )
    node_a = result_a["node"]
    INTERNAL_NODE_A = node_a["node_token"]
    DOC_A_ID = node_a["obj_token"]
    print(f"[OK] 节点A创建成功")
    print(f"     node_token: {INTERNAL_NODE_A}")
    print(f"     obj_token: {DOC_A_ID}")
    print(f"     链接: {FEISHU_DOMAIN}/wiki/{INTERNAL_NODE_A}")

    # 节点B
    result_b = client.create_wiki_node(
        space_id=USER_WIKI_SPACE_ID,
        title=f"内部文档B-{TIMESTAMP}",
        obj_type="docx"
    )
    node_b = result_b["node"]
    INTERNAL_NODE_B = node_b["node_token"]
    DOC_B_ID = node_b["obj_token"]
    print(f"[OK] 节点B创建成功")
    print(f"     node_token: {INTERNAL_NODE_B}")
    print(f"     obj_token: {DOC_B_ID}")
    print(f"     链接: {FEISHU_DOMAIN}/wiki/{INTERNAL_NODE_B}")
except Exception as e:
    print(f"[FAIL] {e}")

### 2.4 对外部迁入的文档追加内容

In [ ]:
print(f"\n[APPEND] 外部迁入文档追加内容")
md_content = f"""
## 迁入后追加的内容

这篇文档从外部迁入 Wiki 知识库后,通过 API 追加了以下内容：

- **迁入时间**: {TIMESTAMP}
- **操作方式**: feishuWikiMoveDoc
- **验证结果**: 外部文档迁入后仍可正常读写
"""

try:
    blocks = md_to_blocks(md_content)
    result = client.api(
        "POST",
        f"/docx/v1/documents/{EXTERNAL_DOC_TOKEN}/blocks/{EXTERNAL_DOC_TOKEN}/children",
        json_data={"children": blocks},
        use_user_token=True
    )
    print(f"[OK] 追加了 {len(result.get('children', []))} 个块")
except Exception as e:
    print(f"[FAIL] {e}")

### 2.5 对内部文档追加内容

In [ ]:
print(f"\n[APPEND] 内部文档A追加内容")
md_a = f"""
## 内部文档A的初始内容

这是在知识库内直接创建的文档节点,通过 API 写入的初始内容. 

| 属性 | 值 |
|------|-----|
| 创建方式 | feishuWikiNodeCreate |
| 节点类型 | docx |
| 时间 | {TIMESTAMP} |
"""

try:
    blocks = md_to_blocks(md_a)
    result = client.api(
        "POST",
        f"/docx/v1/documents/{DOC_A_ID}/blocks/{DOC_A_ID}/children",
        json_data={"children": blocks},
        use_user_token=True
    )
    print(f"[OK] 文档A追加了 {len(result.get('children', []))} 个块")
except Exception as e:
    print(f"[FAIL] {e}")

### 2.6 创建更多文档节点(用于层级测试)

In [ ]:
print(f"\n[CREATE MORE] 创建层级测试节点")
try:
    # 节点C - 将作为父节点
    result_c = client.create_wiki_node(
        space_id=USER_WIKI_SPACE_ID,
        title=f"父节点C-{TIMESTAMP}",
        obj_type="docx"
    )
    INTERNAL_NODE_C = result_c["node"]["node_token"]
    DOC_C_ID = result_c["node"]["obj_token"]
    print(f"[OK] 父节点C: {INTERNAL_NODE_C}")

    # 子节点1
    result_1 = client.create_wiki_node(
        space_id=USER_WIKI_SPACE_ID,
        title=f"子节点1-{TIMESTAMP}",
        obj_type="docx"
    )
    CHILD_1 = result_1["node"]["node_token"]
    print(f"[OK] 子节点1: {CHILD_1}")

    # 子节点2
    result_2 = client.create_wiki_node(
        space_id=USER_WIKI_SPACE_ID,
        title=f"子节点2-{TIMESTAMP}",
        obj_type="docx"
    )
    CHILD_2 = result_2["node"]["node_token"]
    print(f"[OK] 子节点2: {CHILD_2}")
except Exception as e:
    print(f"[FAIL] {e}")

### 2.7 将文档移动为子文档(层级结构)

In [ ]:
print(f"\n[MOVE] 构建层级结构")
try:
    # 将外部迁入的文档移动到父节点C下
    client.move_wiki_node(
        USER_WIKI_SPACE_ID,
        EXTERNAL_NODE_TOKEN,
        parent_node_token=INTERNAL_NODE_C
    )
    print(f"[OK] 外部文档 → 父节点C下")

    # 将内部文档A移动到父节点C下
    client.move_wiki_node(
        USER_WIKI_SPACE_ID,
        INTERNAL_NODE_A,
        parent_node_token=INTERNAL_NODE_C
    )
    print(f"[OK] 内部文档A → 父节点C下")

    # 将子节点1移动到父节点C下
    client.move_wiki_node(
        USER_WIKI_SPACE_ID,
        CHILD_1,
        parent_node_token=INTERNAL_NODE_C
    )
    print(f"[OK] 子节点1 → 父节点C下")

    # 将子节点2移动到子节点1下(三级层级)
    client.move_wiki_node(
        USER_WIKI_SPACE_ID,
        CHILD_2,
        parent_node_token=CHILD_1
    )
    print(f"[OK] 子节点2 → 子节点1下(三级层级)")
except Exception as e:
    print(f"[FAIL] {e}")

### 2.8 验证节点列表(观察层级结构)

In [ ]:
print(f"\n[VERIFY] 知识库节点层级")
def print_tree(space_id, parent_token=None, indent=0):
    """递归打印节点树"""
    nodes = client.list_wiki_nodes(space_id, parent_node_token=parent_token)
    for n in nodes:
        prefix = "  " * indent + "- "
        title = n.get('title', '?')
        token = n.get('node_token', '?')[:15]
        print(f"{prefix}{title} ({token}...)")
        # 递归打印子节点
        print_tree(space_id, n.get('node_token'), indent + 1)

try:
    print("\n[知识库层级结构]")
    print_tree(USER_WIKI_SPACE_ID)
except Exception as e:
    print(f"[FAIL] {e}")

### 2.9 验证子节点内容可读写

In [ ]:
print(f"\n[APPEND] 三级子节点追加内容")
md_child = f"""
## 三级子节点内容

我是被移动到子节点1下的子节点2,验证了 Wiki 知识库支持多级层级结构. 

- 层级: 3级
- 父节点: 子节点1
- 祖父节点: 父节点C
"""

# 获取子节点2的文档ID
child2_doc_id = result_2["node"]["obj_token"]

try:
    blocks = md_to_blocks(md_child)
    result = client.api(
        "POST",
        f"/docx/v1/documents/{child2_doc_id}/blocks/{child2_doc_id}/children",
        json_data={"children": blocks},
        use_user_token=True
    )
    print(f"[OK] 子节点2追加了 {len(result.get('children', []))} 个块")
except Exception as e:
    print(f"[FAIL] {e}")

## 第三部分：清理(可选)

> ⚠️ 以下 cell 会删除创建的知识库和文档. 如需保留,请勿运行. 

In [ ]:
# print(f"\n[CLEANUP] 删除测试数据")
# try:
#     # 删除知识库(会连带删除所有节点)
#     client.api("DELETE", f"/wiki/v2/spaces/{USER_WIKI_SPACE_ID}", use_user_token=True)
#     print(f"[OK] 知识库已删除: {USER_WIKI_SPACE_ID}")
# except Exception as e:
#     print(f"[FAIL] {e}")